# Input-mount probe (v6: strip the `trunk.` prefix and check it loads clean)

`enable_gpu: false` - no GPU quota.

v5 showed the mismatch is a pure prefix difference:

```
checkpoint:  trunk.cls_token   trunk.pos_embed   trunk.blocks.0.attn.qkv.weight
plain timm:  cls_token         pos_embed         blocks.0.attn.qkv.weight
```

and that `forward_features(x)` returns `(B, 197, 768)` - a token sequence, which is exactly
what the existing `Model` wrapper consumes (CLS + patch tokens).

If stripping `trunk.` leaves nothing missing, the BiomedCLIP vision tower can be built from
the **preinstalled timm** with no `open_clip` and no internet at scoring time. That is the
go/no-go for the whole "swap the vision encoder" line of work.

This cell also reports the preprocessing constants, because the checkpoint's are CLIP-style
and differ from the ImageNet ones currently hardcoded in `Model`.


In [ ]:
import json
from pathlib import Path

import timm
import torch

MODEL_DIR = Path("/kaggle/input/models/mrquadian/biomedclip/pytorch/default/1")

raw = torch.load(MODEL_DIR / "open_clip_pytorch_model.bin", map_location="cpu")
sd = raw.get("state_dict", raw) if isinstance(raw, dict) else raw
vis = {k[len("visual."):]: v for k, v in sd.items() if k.startswith("visual.")}

print("=== non-trunk keys inside visual.* (if any) ===")
odd = [k for k in vis if not k.startswith("trunk.")]
print(" ", odd if odd else "(none)")

stripped = {k[len("trunk."):]: v for k, v in vis.items() if k.startswith("trunk.")}
print(f"\nstripped tensors: {len(stripped)}")

model = timm.create_model("vit_base_patch16_224", pretrained=False, num_classes=0)
missing, unexpected = model.load_state_dict(stripped, strict=False)
print(f"timm params : {len(model.state_dict())}")
print(f"missing     : {len(missing)}  {list(missing)[:10]}")
print(f"unexpected  : {len(unexpected)}  {list(unexpected)[:10]}")

print()
print("=== sanity: are the loaded weights actually the checkpoint's, not random init? ===")
w = model.patch_embed.proj.weight
print(f"  patch_embed.proj.weight  mean={w.mean():+.6f} std={w.std():.6f}")
print(f"  cls_token                mean={model.cls_token.mean():+.6f} std={model.cls_token.std():.6f}")
norm_w = model.blocks[0].attn.qkv.weight
print(f"  blocks.0.attn.qkv.weight mean={norm_w.mean():+.6f} std={norm_w.std():.6f}")

model.eval()
with torch.no_grad():
    tok = model.forward_features(torch.zeros(2, 3, 224, 224))
print(f"\n  forward_features -> {tuple(tok.shape)}")
print(f"  CLS = tok[:, 0].shape {tuple(tok[:, 0].shape)}")
print(f"  patches = tok[:, 1:].shape {tuple(tok[:, 1:].shape)}")

print()
print("=== preprocessing this checkpoint expects (differs from the current ImageNet constants) ===")
cfg = None
for p in MODEL_DIR.glob("*.json"):
    if "tokenizer" in p.name or "special_tokens" in p.name:
        continue
    try:
        cfg = json.loads(p.read_text())
    except Exception:
        pass
if cfg and "preprocess_cfg" in cfg:
    pp = cfg["preprocess_cfg"]
    print(f"  mean = {pp.get('mean')}")
    print(f"  std  = {pp.get('std')}")
    print("  (Model currently hardcodes ImageNet 0.485/0.456/0.406 and 0.229/0.224/0.225)")

ok = (len(missing) == 0)
print()
print("VERDICT:", "GO - vision tower loads cleanly from preinstalled timm; no open_clip, "
      "no internet needed at scoring time" if ok else
      f"NO-GO as-is - {len(missing)} tensors unmatched")
